# ChaseEscapeEnv – 3 models, team rewards, strategy planner (Colab)

**Goal:** Police + enemy + **strategy planner** (3 agents). Strategy coordinates police: go to place, trap, bottleneck, block exit. **Team shared rewards**; supports multi police/enemy later (1v1 first). Three zips for higher accuracy.

**Step 1 – Train police (enemy static)**  
Map + agents (drawn or random). Enemy does **not** move. Only police is trained to find and arrest. → `ppo_chase_escape_final.zip`.

**Step 2 – Train enemy (police = trained model)**  
Load police zip; train only enemy to reach escape zone (with walls). → `ppo_enemy_escape_final.zip`.

**Step 3 – Train strategy planner (3rd model)**  
Strategy planner is the RL agent. Commands police: chase, block exit, bottleneck, hold, flank. **Team shared reward** (arrest +, escape −). Enemy scripted or loaded. → `ppo_strategy_final.zip`.

**Run all 3:** Strategy sees state → outputs command; police move by strategy; enemy uses enemy model. No training, full coordination.

In [ ]:
# Clone the pursuit_arena repository in Colab and move into the project folder.
# This uses your GitHub repo: https://github.com/GraslyDias/pursuit_arena.git

%cd /content
!git clone https://github.com/GraslyDias/pursuit_arena.git
%cd /content/pursuit_arena

In [ ]:
# Install dependencies in Colab only (NOT on your local machine).

!pip install pygame gymnasium stable-baselines3[extra] numpy

In [ ]:
# Optional: quick environment check
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv
from stable_baselines3.common.env_checker import check_env

env = ChaseEscapeEnv(render_mode=None)
check_env(env, warn=True)
env.close()

In [ ]:
# Step 1: Train POLICE only (enemy does not move – static). Random or saved map.
# Upload training_map.json to use your drawn map + agent positions; else random walls + random positions.
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

def make_env():
    env = ChaseEscapeEnv(render_mode=None)
    env.static_enemy = True   # Enemy does not move; only police is trained to find and arrest
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
        print("Training on saved map: training_map.json")
    env = Monitor(env, str(log_dir / "monitor.csv"))
    return env

vec_env = DummyVecEnv([make_env])

# Use CPU for PPO (MlpPolicy); GPU gives poor utilization and can be slower for this policy.
# Re-train from current zip to improve (continue training). If no zip exists, train from scratch.
model_path = log_dir / "ppo_chase_escape_final.zip"
if model_path.exists():
    model = PPO.load(str(model_path), env=vec_env, device="cpu")
    print("Loaded existing model for continued training.")
else:
    model = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb"), device="cpu")
    print("Training from scratch.")
model.learn(total_timesteps=300_000)
model.save(str(model_path))
print("Saved improved model to", model_path)
vec_env.close()

## Step 2: Train **enemy** only (police = trained model, not trained here)

Run this **after** Step 1. Police is the **trained** model (loaded from zip); only the enemy is trained to escape to the escape zone. Use your drawn map (training_map.json with walls). Saves `ppo_enemy_escape_final.zip`.

In [ ]:
# Step 2: Train ENEMY only. Police = trained model (load zip). Use map with walls (training_map.json).
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnemyEnv, load_training_map

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

# Load trained police model (do not train it)
police_zip = log_dir / "ppo_chase_escape_final.zip"
assert police_zip.exists(), "Run Step 1 first to create ppo_chase_escape_final.zip"
police_model = PPO.load(str(police_zip), device="cpu")

def make_env_enemy():
    env = ChaseEscapeEnemyEnv(render_mode=None, police_model=police_model)
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
        print("Enemy training on saved map (with walls): training_map.json")
    env = Monitor(env, str(log_dir / "monitor_enemy.csv"))
    return env

vec_env = DummyVecEnv([make_env_enemy])

model_path_enemy = log_dir / "ppo_enemy_escape_final.zip"
if model_path_enemy.exists():
    model_enemy = PPO.load(str(model_path_enemy), env=vec_env, device="cpu")
    print("Loaded existing enemy model for continued training.")
else:
    model_enemy = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_enemy"), device="cpu")
    print("Training enemy from scratch.")
model_enemy.learn(total_timesteps=300_000)
model_enemy.save(str(model_path_enemy))
print("Saved enemy model to", model_path_enemy)
vec_env.close()

## Step 3: Train strategy planner (3rd model – coordinate police, trap, bottleneck)

Strategy planner is the RL agent. It sees global state and outputs **commands** for police: chase, block exit, bottleneck, hold, flank. **Team shared reward** (arrest +, escape −). Enemy can be scripted or your trained enemy model. Saves `ppo_strategy_final.zip`. Run after Step 1 and Step 2.

In [ ]:
# Step 3: Train STRATEGY PLANNER (3rd model). Team shared rewards; strategy commands police.
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeStrategyEnv, load_training_map

log_dir = Path("runs/ppo_chase_escape")
log_dir.mkdir(parents=True, exist_ok=True)

# Optional: use trained enemy so strategy learns vs strong opponent
enemy_zip = log_dir / "ppo_enemy_escape_final.zip"
enemy_model = PPO.load(str(enemy_zip), device="cpu") if enemy_zip.exists() else None

def make_env_strategy():
    env = ChaseEscapeStrategyEnv(render_mode=None, enemy_model=enemy_model)
    if Path("training_map.json").exists():
        env.training_map_options = load_training_map("training_map.json")
        print("Strategy training on saved map: training_map.json")
    env = Monitor(env, str(log_dir / "monitor_strategy.csv"))
    return env

vec_env = DummyVecEnv([make_env_strategy])
model_path_strategy = log_dir / "ppo_strategy_final.zip"
if model_path_strategy.exists():
    model_strategy = PPO.load(str(model_path_strategy), env=vec_env, device="cpu")
    print("Loaded existing strategy model for continued training.")
else:
    model_strategy = PPO("MlpPolicy", vec_env, verbose=1, tensorboard_log=str(log_dir / "tb_strategy"), device="cpu")
    print("Training strategy from scratch.")
model_strategy.learn(total_timesteps=300_000)
model_strategy.save(str(model_path_strategy))
print("Saved strategy model to", model_path_strategy)
vec_env.close()

## Run all 3 models (strategy commands police, enemy uses trained model)

Strategy planner sees state and outputs commands; police move according to strategy. Enemy uses trained enemy model. No training – evaluation only.

In [ ]:
# Run all 3: strategy + enemy models (strategy commands police)
from pathlib import Path

from stable_baselines3 import PPO

from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeStrategyEnv, load_training_map

log_dir = Path("runs/ppo_chase_escape")
options = load_training_map("training_map.json") if Path("training_map.json").exists() else None
enemy_model = PPO.load(str(log_dir / "ppo_enemy_escape_final.zip"), device="cpu") if (log_dir / "ppo_enemy_escape_final.zip").exists() else None
strategy_model = PPO.load(str(log_dir / "ppo_strategy_final.zip"), device="cpu")

env = ChaseEscapeStrategyEnv(render_mode=None, enemy_model=enemy_model)
if options is not None:
    env.training_map_options = options
    print("Using map: training_map.json")

episodes = 5
for ep in range(episodes):
    obs, _ = env.reset()
    done, truncated = False, False
    ep_rew = 0.0
    while not (done or truncated):
        action, _ = strategy_model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(int(action))
        ep_rew += reward
    print(f"Episode {ep+1}: reward={ep_rew:.2f}, {info}")
env.close()

In [ ]:
# Evaluate the trained model in Colab (no rendering window)
from stable_baselines3 import PPO
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv

env = ChaseEscapeEnv(render_mode=None)
model = PPO.load("runs/ppo_chase_escape/ppo_chase_escape_final.zip", env=env)

episodes = 5
for ep in range(episodes):
    obs, info = env.reset()
    done = False
    truncated = False
    ep_reward = 0.0
    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        ep_reward += float(reward)
    print(f"Episode {ep+1}: reward={ep_reward:.2f}, info={info}")

env.close()

In [ ]:
# Download trained models to your local machine (from Colab)
from google.colab import files

files.download("runs/ppo_chase_escape/ppo_chase_escape_final.zip")   # police
files.download("runs/ppo_chase_escape/ppo_enemy_escape_final.zip")   # enemy
files.download("runs/ppo_chase_escape/ppo_strategy_final.zip")      # strategy planner